# Cognito User Pool Migration Script

This notebook provides a comprehensive migration process for moving users from one Cognito User Pool to another while maintaining data consistency across S3 and DynamoDB.

## Overview
- **Export users** from the old Cognito pool
- **Import users** to the new pool and create sub ID mapping
- **Migrate S3 folders** from old sub-based paths to new sub-based paths
- **Update DynamoDB records** to use the new sub IDs

## Prerequisites
- AWS credentials configured
- Access to old and new Cognito User Pools
- S3 bucket access
- DynamoDB table access

# Setup and Configuration

Configure all necessary variables and imports for the migration process.

In [ ]:
%pip install boto3==1.40.45

In [ ]:
import os
import time
import json
from typing import Dict, List
from boto3.session import Session

OLD_POOL_ID = ''
NEW_POOL_ID = ''
S3_BUCKET = ''
DYNAMODB_TABLE = ''
TEMP_PASSWORD = ''
PROFILE = ''
REGION = 'us-east-1'

session = Session(profile_name=PROFILE)
cognito = session.client('cognito-idp', region_name=REGION)
s3 = session.client('s3', region_name=REGION)
dynamodb = session.resource('dynamodb', region_name=REGION)
table = dynamodb.Table(DYNAMODB_TABLE)

sub_mapping: Dict[str, str] = {}

os.makedirs('cognito_migration', exist_ok=True)

# Step 1: Export Users

Export all users from the old Cognito User Pool to a JSON file for backup and processing.

In [ ]:
def export_users(old_pool_id: str, output_file: str = 'cognito_migration/users_export.json'):
    """Export all users from the old pool"""
    print(f"Exporting users from pool: {old_pool_id}")
    users = []
    pagination_token = None

    while True:
        if pagination_token:
            response = cognito.list_users(
                UserPoolId=old_pool_id,
                PaginationToken=pagination_token,
                Limit=60
            )
        else:
            response = cognito.list_users(
                UserPoolId=old_pool_id,
                Limit=60
            )

        users.extend(response['Users'])

        if 'PaginationToken' not in response:
            break
        pagination_token = response['PaginationToken']
        time.sleep(0.2)  # Rate limiting

    print(f"Exported {len(users)} users")

    with open(output_file, 'w') as f:
        json.dump(users, f, indent=2, default=str)

    return users

users = export_users(OLD_POOL_ID)

# Step 2: Import Users and Create Sub Mapping

Import users to the new Cognito pool while creating a mapping between old and new subject (sub) IDs.

In [ ]:
def import_users(users: List[Dict], new_pool_id: str, temp_password: str, output_file: str = 'cognito_migration/sub_mapping.json'):
    """Import users into new pool and track sub mapping"""
    print(f"Importing {len(users)} users to pool: {new_pool_id}")

    for i, user in enumerate(users):
        try:
            old_sub = next(
                (attr['Value'] for attr in user['Attributes'] if attr['Name'] == 'sub'),
                None
            )

            # Find email attribute to use as username
            email_attr = next(
                (attr['Value'] for attr in user['Attributes'] if attr['Name'] == 'email'),
                None
            )

            if not email_attr:
                print(f"Skipping user {user['Username']}: no email attribute found")
                continue

            username = email_attr

            attributes = [
                attr for attr in user['Attributes']
                if attr['Name'] not in ['sub']
            ]

            response = cognito.admin_create_user(
                UserPoolId=new_pool_id,
                Username=username,
                UserAttributes=attributes,
                TemporaryPassword=temp_password,
                MessageAction='SUPPRESS'
            )

            new_sub = next(
                (attr['Value'] for attr in response['User']['Attributes']
                 if attr['Name'] == 'sub'),
                None
            )

            if old_sub and new_sub:
                sub_mapping[old_sub] = new_sub
                print(f"[{i+1}/{len(users)}] Migrated {username}: {old_sub} -> {new_sub}")
            if user['UserStatus'] == 'CONFIRMED':
                cognito.admin_set_user_password(
                    UserPoolId=new_pool_id,
                    Username=username,
                    Password=temp_password,
                    Permanent=False
                )

            time.sleep(0.1)

        except Exception as e:
            print(f"Error importing user {username}: {str(e)}")

    with open(output_file, 'w') as f:
        json.dump(sub_mapping, f, indent=2)

    print(f"Saved sub mapping for {len(sub_mapping)} users")
    return sub_mapping

sub_mapping = import_users(users, NEW_POOL_ID, TEMP_PASSWORD)

# Step 3: Migrate S3 Folders

Move S3 objects from old sub-based folder paths to new sub-based folder paths.

In [ ]:
def migrate_s3_folders(s3_bucket: str, sub_mapping: Dict[str, str]):
    """Migrate S3 folders from /artwork/{old_sub} to /artwork/{new_sub}"""
    print(f"Migrating S3 folders in bucket: {s3_bucket}")

    for old_sub, new_sub in sub_mapping.items():
        old_prefix = f"artwork/{old_sub}/"
        new_prefix = f"artwork/{new_sub}/"

        try:
            paginator = s3.get_paginator('list_objects_v2')
            pages = paginator.paginate(Bucket=s3_bucket, Prefix=old_prefix)

            objects_moved = 0
            for page in pages:
                if 'Contents' not in page:
                    continue

                for obj in page['Contents']:
                    old_key = obj['Key']
                    new_key = old_key.replace(old_prefix, new_prefix, 1)

                    copy_source = {'Bucket': s3_bucket, 'Key': old_key}
                    s3.copy_object(
                        CopySource=copy_source,
                        Bucket=s3_bucket,
                        Key=new_key
                    )

                    s3.delete_object(Bucket=s3_bucket, Key=old_key)
                    objects_moved += 1

            if objects_moved > 0:
                print(f"Moved {objects_moved} objects: {old_prefix} -> {new_prefix}")

        except Exception as e:
            print(f"Error migrating S3 folder for {old_sub}: {str(e)}")

    print("S3 migration complete")

migrate_s3_folders(S3_BUCKET, sub_mapping)

# Step 4: Migrate DynamoDB Records

Update all DynamoDB records to use the new sub IDs based on the mapping created during user import.

In [ ]:
def migrate_dynamodb_records(dynamodb_table, sub_mapping: Dict[str, str], user_id_field: str = 'userId'):
    """Update DynamoDB records with new sub values using batch writes"""
    print(f"Migrating DynamoDB table: {dynamodb_table}")

    scan_kwargs = {}
    items_to_update = []
    items_scanned = 0

    while True:
        response = table.scan(**scan_kwargs)
        items = response.get('Items', [])
        items_scanned += len(items)

        for item in items:
            old_sub = item.get(user_id_field)

            if old_sub and old_sub in sub_mapping:
                item[user_id_field] = sub_mapping[old_sub]
                items_to_update.append(item)

        if 'LastEvaluatedKey' not in response:
            break
        scan_kwargs['ExclusiveStartKey'] = response['LastEvaluatedKey']

    print(f"Scanned {items_scanned} items, found {len(items_to_update)} to update")

    items_updated = 0
    batch_size = 25

    for i in range(0, len(items_to_update), batch_size):
        batch = items_to_update[i:i + batch_size]

        try:
            with table.batch_writer() as writer:
                for item in batch:
                    writer.put_item(Item=item)
                    items_updated += 1

            print(f"Batch write progress: {items_updated}/{len(items_to_update)}")
            time.sleep(0.1)

        except Exception as e:
            print(f"Error in batch write: {str(e)}")
            for item in batch:
                try:
                    table.put_item(Item=item)
                    items_updated += 1
                except Exception as e2:
                    print(f"Error updating individual item: {str(e2)}")

    print(f"Updated {items_updated} DynamoDB records")

migrate_dynamodb_records(table, sub_mapping)

# Step 5: Load Saved Mapping (Optional)

Load a previously saved sub mapping file if you need to resume or rerun parts of the migration.

In [ ]:
def load_mapping(mapping_file: str = 'cognito_migration/sub_mapping.json'):
    """Load previously saved mapping"""
    with open(mapping_file, 'r') as f:
        sub_mapping = json.load(f)
    print(f"Loaded mapping for {len(sub_mapping)} users")
    return sub_mapping

sub_mapping = load_mapping()